In [1]:
import os
import sys

llava_path = "/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA"
sys.path.append(llava_path)
from llava.eval.run_llava import eval_model

[2025-03-24 10:37:46,591] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [ ]:
import os
os.chdir('/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA-Med')


from peft import PeftModel
from huggingface_hub import create_repo

import sys
import warnings
warnings.filterwarnings("ignore")
import random
import torch
from torch.utils.data.dataset import Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import io
import requests
from datetime import datetime
import gc
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import json
import time 
from collections import defaultdict 
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, LoraModel, get_peft_model, prepare_model_for_kbit_training
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import Conversation
from llava.mm_utils import tokenizer_image_token, process_images
from llava.model.builder import load_pretrained_model
from llava.conversation import conv_templates
from tqdm import tqdm
from torch.utils.data import DataLoader
import glob
import re 
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from multiprocessing import Pool, cpu_count
import pydicom
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import uuid
import pickle
from transformers import LlamaTokenizer
from llava.model import LlavaMistralForCausalLM  # LLaVA-Med의 모델 클래스 import
from dotenv import load_dotenv
import wandb
import fnmatch
from functools import lru_cache
from torchvision.transforms import Resize, Compose, ToTensor
from functools import partial  # collate_fn에 인자 전달을 위한 패키지
from torch.utils.data._utils.pin_memory import pin_memory
from transformers import AutoTokenizer, AutoModelForCausalLM
from llava.utils import disable_torch_init
from accelerate import init_empty_weights
from accelerate import Accelerator, DeepSpeedPlugin
from transformers import BitsAndBytesConfig
import cv2  # OpenCV를 활용한 빠른 이미지 저장
from huggingface_hub import notebook_login
from accelerate.utils import set_module_tensor_to_device 
from transformers.integrations import WandbCallback
from transformers.models.mistral.modeling_mistral import MistralRotaryEmbedding
import shutil
from transformers.integrations.deepspeed import HfTrainerDeepSpeedConfig
from langgraph.graph import END, StateGraph
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache 
from typing import TypedDict, List, Dict, Any, Annotated
import operator
from llava.conversation import conv_templates, SeparatorStyle
from transformers import StoppingCriteria
from llava.utils import disable_torch_init
from enum import auto, Enum
from contextlib import redirect_stdout
load_dotenv()
set_llm_cache(InMemoryCache())    
load_dotenv()
os.environ["WANDB_API_KEY"] = ""
os.environ["HUGGING_FACE_HUB_TOKEN"] = ""
notebook_login()
wandb.login()

# CUDA 환경 변수 설정 - 메모리 초과 문제 방지
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# 사용 가능한 GPU 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
device_count = torch.cuda.device_count()
if device_count > 1:
    print(f"🖥 {device_count}개 GPU 사용 중: {list(range(device_count))}")
    
# 캐시 디렉토리 설정  
CACHE_DIR = "/home/cbn-gpu12/FNF/VLM/LLavA/dataset/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# 장치 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sarahyo941 (sarahyo941-university-of-ulsan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


🖥 4개 GPU 사용 중: [0, 1, 2, 3]


In [3]:
def get_img_path(serial, side):
    base_path = '/home/cbn-gpu12/FNF/VLM/LLaVA/dataset/preprocessed_data'
    img_filename = f'{serial}.png'
    if side == 'AP':
        img_path = os.path.join(base_path,'Figure5_AP', img_filename)
    elif side == 'Lateral':
        img_path = os.path.join(base_path,'Figure5_LAT', img_filename)
    
    
    return img_path 

def process_and_save( source_dir, output_folder):
 
    json_data_list = []
    metadata_path = os.path.join(source_dir, 'metadata.json')
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r', encoding='utf-8') as f:
            try: 
                metadata_str = f.read().strip()
                metadata = json.loads(metadata_str)
                if not isinstance(metadata, list):
                    metadata = []
            except json.JSONDecodeError:
                metadata.data = []
    else:
        metadata = []
    for item in metadata:
        img_path = get_img_path( item['serial'], item['side'])
        image = Image.open(os.path.join(img_path)).convert('RGB')
    
        image = img_path

        unique_id = str(uuid.uuid4())



        answers = item['answer']
        formatted_answers = "".join(answers)

        json_data = {
            'id': unique_id,
            'image': f"{item['serial']}.png",
            'side': f"{item['side']}",
            'label': f"{item['label']}",
            'LR': f"{item['LR']}",
            'conversations':[
                {
                    'from': 'human',
                    'value': item['question']
                },
                {
                    'from':'llava-med',
                    'value': formatted_answers
                }
            ]
        }

        json_data_list.append(json_data)


    json_output_path = os.path.join(output_folder, 'dataset.json')
    with open(json_output_path,'w') as json_file:
        json.dump(json_data_list, json_file, indent=4)


def save_dataset(source_dir, output_folder):

    process_and_save(source_dir, output_folder)

In [4]:
output_folder = '/mnt/nas_backup/고효진/FNF/dataset/detected_qa/finetuned'
source_dir = '/mnt/nas_backup/고효진/FNF/dataset/detected_qa'

save_dataset(source_dir, output_folder)
save_dataset(source_dir, output_folder)



In [5]:
import json
from collections import defaultdict

# JSON 파일 경로
file_path = '/mnt/nas_backup/고효진/FNF/dataset/detected_qa/metadata.json'

# JSON 데이터 불러오기
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# serial, side, LR 기준으로 그룹화
grouped = defaultdict(list)
for item in data:
    key = (item['serial'], item['side'], item['LR'])
    grouped[key].append(item)

# 중복되는 키를 가진 항목들만 추출
duplicates = []
for key, items in grouped.items():
    if len(items) > 1:
        duplicates.extend(items)

# 결과 출력 (필요하면 파일로 저장 가능)
print(f"중복된 항목 수: {len(duplicates)}")
for item in duplicates:
    print(item)

# 또는 결과를 JSON 파일로 저장하고 싶다면 아래 코드 사용
# with open('duplicates.json', 'w', encoding='utf-8') as f:
#     json.dump(duplicates, f, indent=2, ensure_ascii=False)


중복된 항목 수: 0


In [6]:
class VQARAD(Dataset):
    def __init__(self, img_src_dir, split):
        super(VQARAD, self).__init__()
        self.split = split
        self.image_folder = img_src_dir
        self.paths = {
            'train': '/mnt/nas_backup/고효진/FNF/dataset/detected_qa/train_metadata.json',
            'test': '/mnt/nas_backup/고효진/FNF/dataset/detected_qa/test_metadata.json'
        }
        with open(self.paths[self.split], 'r') as f:
            self.dataset = json.load(f)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        id = item['serial']
        question = item['question']
        answer = item['answer']
        image_path = get_img_path(id, item['side'])
        image = Image.open(os.path.join(self.image_folder, image_path)).convert('RGB')
    

        return id, question, answer, image
        
        

In [7]:
class DataCollator:
    def __init__(self, tokenizer, split, conversation_template, pad_token_id, image_processor, model_config):
        self.tokenizer = tokenizer 
        self.split = split 
        self.conversation_template = conversation_template
        self.pad_token_id = pad_token_id
        self.image_processor = image_processor
        self.model_config = model_config

    def __call__(self, rows):
        if self.split == 'train':
            return self._collate_train(rows)
        elif self.split == 'test':
            return self._collate_test(rows)
        else:
            return ValueError(f'Invalid split: {self.split}')

    def _collate_train(self, rows):
        train_input_ids_list = []
        train_labels_list = []
        train_images = []

        for row in rows: 
            id, question, answer, image = row
            train_images.append(image)

            question = question.replace(DEFAULT_IMAGE_TOKEN,'').strip()
            question = DEFAULT_IMAGE_TOKEN + '\n' + question

            conv = self.conversation_template.copy()
            conv.append_message(conv.roles[0], question)
            conv.append_message(conv.roles[1], None)
            prefix = conv.get_prompt()

            conv = self.conversation_template.copy()
            conv.append_message(conv.roles[0], question)
            conv.append_message(conv.roles[1], answer)
            full = conv.get_prompt()

            prefix = tokenizer_image_token(prefix, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt')
            full = tokenizer_image_token(full, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")

            prefix_length = prefix.size(0)
            full_length = full.size(0)

            train_input_ids = full
            train_labels = full.clone()
            train_labels[:prefix_length] = -100

            train_input_ids_list.append(train_input_ids)
            train_labels_list.append(train_labels)

        pad_value = -114514
        train_input_ids = pad_sequence(train_input_ids_list, batch_first = True, padding_value=pad_value)
        train_labels = pad_sequence(train_labels_list, batch_first=True, padding_value=pad_value)
        train_attention_mask = (train_input_ids != pad_value).long()

        train_input_ids[train_input_ids == pad_value] = self.pad_token_id
        train_labels[train_labels == pad_value] = self.pad_token_id

        train_images = process_images(train_images, self.image_processor, self.model_config).to(torch.bfloat16)

        return {
            'input_ids': train_input_ids,
            'labels': train_labels,
            'attention_mask': train_attention_mask,
            'images': train_images
        }
    def  _collate_test(self, rows):
        pass
        
            

In [ ]:
hf_token =''
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path="microsoft/llava-med-v1.5-mistral-7b",
    model_base=None,
    model_name="llava-med-v1.5-mistral-7b",
    token=hf_token,
    use_flash_attn=False
)
image_source_dir = '/home/cbn-gpu12/FNF/VLM/LLaVA/dataset/preprocessed_data'
vqa_rad_dataset_train = VQARAD(image_source_dir,split='train')
vqa_rad_test = VQARAD(image_source_dir, split='test')

conv = conv_templates['mistral_instruct']

collate_fn = DataCollator(
    tokenizer = tokenizer,
    split = 'train',
    conversation_template=conv,
    pad_token_id=tokenizer.pad_token_id,
    image_processor=image_processor,
    model_config=model.config
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.la

In [ ]:
class ModelTrainer:
    def __init__(self):
        pass

    def train(self, state: LLaVAMedPipelineState) -> LLaVAMedPipelineState:
        """모델 학습 및 저장"""
        print("===== 모델 학습 =====")

        # Wandb 초기화
        wandb.init(
            project="fnf-classification",
            name=f"fnf-classification-{datetime.now().strftime('%Y%m%d-%H%M')}",
            config={
                "model_name": state["model_path"],
                "learning_rate": 2e-5,
                "epochs": 5,
                "batch_size": 1,
                "gradient_accumulation_steps": 4,
                "lora_r": 8,
                "lora_alpha": 16
            }
        )

        # 양자화 모델을 학습 가능한 상태로 준비
        print("양자화 모델을 학습 가능한 상태로 준비 중...")
        model = state["model"]

        # LoRA 설정
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["k_proj", "q_proj", "v_proj", "out_proj"],
            bias="none",
            task_type="CAUSAL_LM"
        )

        # LoRA 모델 생성
        peft_model = get_peft_model(model, lora_config)
        peft_model.config.use_cache = False
        peft_model.print_trainable_parameters()

        # 학습 설정
        training_args = TrainingArguments(
            output_dir=state["output_dir"],
            report_to="wandb",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            logging_steps=5,
            learning_rate=2e-5,
            logging_strategy="epoch",
            save_strategy="epoch",
            num_train_epochs=5,
            warmup_ratio=0.03,
            weight_decay=0.01,
            remove_unused_columns=True,
            gradient_checkpointing=True,
            fp16=True,
            bf16=False,
            optim='paged_adamw_8bit',
            deepspeed={
                "train_batch_size": "auto",
                "zero_optimization": {
                    "stage": 3,
                    "overlap_comm": True,
                    "contiguous_gradients": True,
                    "reduce_bucket_size": "auto",
                    "stage3_prefetch_bucket_size": "auto",
                    "stage3_param_persistence_threshold": "auto"
                },
                "fp16": {
                    "enabled": True
                },
                "zero_allow_untested_optimizer": True
            }
        )

        # Trainer 초기화
        trainer = Trainer(
            model=peft_model,
            args=training_args,
            train_dataset=state["vqa_rad_dataset_train"],
            data_collator=state["collate_fn"]
        )

        # 학습 실행
        print("학습 시작...")
        trainer.train()
        print("학습 완료!")

        # 모델 저장
        lora_save_path = os.path.join(state["output_dir"], "lora_trained_model")
        trainer.save_model(lora_save_path)
        print(f"LoRA 모델 저장 완료: {lora_save_path}")

        # 모델 병합 및 저장
        try:
            # GPU 메모리 정리
            torch.cuda.empty_cache()
            gc.collect()

            # 기본 모델 로드
            print("기본 모델 로드 중...")
            base_model = AutoModelForCausalLM.from_pretrained(state["model_path"], torch_dtype=torch.float16)

            # 학습된 모델 병합
            print("학습된 모델 병합 중...")
            trained_model = PeftModel.from_pretrained(base_model, lora_save_path)
            merged_trained_model = trained_model.merge_and_unload()

            # 병합 모델 저장 및 Hugging Face Hub에 업로드
            merged_save_path = os.path.join(state["output_dir"], "merged_trained_model")
            merged_trained_model.save_pretrained(
                merged_save_path,
                push_to_hub=True,
                repo_id="sarahyo941/llava-med-v1.5-mistral-7b-oo"
            )
            state["tokenizer"].save_pretrained(
                merged_save_path,
                push_to_hub=True,
                repo_id="sarahyo941/llava-med-v1.5-mistral-7b-oo"
            )

            print(f"모델 병합 및 저장 완료: {merged_save_path}")

        except Exception as e:
            print(f"모델 병합 오류: {e}")
            import traceback
            traceback.print_exc()

        finally:
            torch.cuda.empty_cache()
            gc.collect()
            if wandb.run is not None:
                wandb.finish()

        return {**state, "training_completed": True}

In [ ]:
trainer.save_model('/mnt/nas_backup/고효진/FNF/dataset/detected_qa/finetuned/lora_trained_model')
base_model = AutoModelForCausalLM.from_pretrained('microsoft/llava-med-v1.5-mistral-7b')

trained_model = PeftModel.from_pretrained(base_model, '/mnt/nas_backup/고효진/FNF/dataset/detected_qa/finetuned/lora_trained_model')
merged_trained_model = trained_model.merge_and_unload()
merged_trained_model.save_pretrained('/mnt/nas_backup/고효진/FNF/dataset/detected_qa/finetuned/merged_trained_model', push_to_hub=True, repo_id='sarahyo941/llava-med-v1.5-mistral-7b-oo')

tokenizer.save_pretrained('/content/drive/MyDrive/merged_trained_model', push_to_hub=True, repo_id='sarahyo941/llava-med-v1.5-mistral-7b-oo')




def safe_prepare_model_for_kbit_training(model):
    try:
        model.train()
        model.gradient_checkpointing_enable()

        # 불필요한 캐시 제거
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        # prepare 단계에서 OOM 가능
        model = prepare_model_for_kbit_training(model)

        # 성공적으로 메모리 준비 완료
        return model

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print("❗ CUDA OOM 오류 발생. 캐시 정리 후 재시도합니다.")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

            # 재시도
            model = prepare_model_for_kbit_training(model)
            return model
        else:
            raise e
            
model.train()
model.gradient_checkpointing_enable()

model = safe_prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r = 8,
    target_modules = ['k_proj','q_proj','v_proj','out_proj'],
    lora_alpha=16
)

peft_model = get_peft_model(model, lora_config, 'default')
peft_model.print_trainable_parameters()



training_args = TrainingArguments(
    output_dir="trained_llava-med",
    report_to="wandb",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=5,
    learning_rate=2e-5,
    logging_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    remove_unused_columns=True,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    deepspeed={
        "train_batch_size": "auto", 
        "zero_optimization": {
            "stage": 3,
            "overlap_comm": True,
            "contiguous_gradients": True,
            "reduce_bucket_size": "auto",
            "stage3_prefetch_bucket_size": "auto",
            "stage3_param_persistence_threshold": "auto"
        },
        "fp16": {
            "enabled": True
        },
        "zero_allow_untested_optimizer": True
    }
)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=vqa_rad_dataset_train,
    data_collator=collate_fn
)

peft_model.config.use_cache=False
trainer.train()
